# Day 18: Project — Complete Attention-Based Text Generator

**Goal:** Assemble Days 11, 14, 15, 16, 17 into one polished language model. This is the Phase 3 capstone.

### What you'll build today

A clean, reusable text generator that:

1. **Tokenizes** text (character-level for now)
2. **Embeds** tokens AND positions (Day 11 + 17)
3. **Attends** with multi-head causal self-attention (Day 15 + 16)
4. **Predicts** next tokens with cross-entropy loss (Day 14)
5. **Generates** text with temperature + top-K sampling
6. **Tracks** train and validation loss (Day 5 + 8)
7. **Saves** model artifacts for reuse (Day 9 + 12)

### Architecture diagram

```
input token IDs       (B, T)
     ▼
token embedding       (B, T, D)
     ▼
+ position embedding  (B, T, D)
     ▼
multi-head causal     (B, T, D)
self-attention
     ▼
output projection     (B, T, V)
     ▼
predicted next token  (sample)
```

5 lines of architecture. Same skeleton scales up to GPT — just deeper.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
import json
import os
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Step 1: A Single `Config` Object

Real projects put all hyperparameters in one place. Change one number → re-run.

In [ ]:
@dataclass
class Config:
    # Architecture
    embed_dim: int = 64
    num_heads: int = 4
    block_size: int = 32       # context window: how many chars the model sees
    
    # Training
    batch_size: int = 32
    learning_rate: float = 1e-3
    steps: int = 3000
    eval_every: int = 200      # how often to compute val loss
    eval_iters: int = 20       # number of batches to average val loss over
    
    # Will be filled in after building the tokenizer
    vocab_size: int = 0

cfg = Config()
print(f"Configuration:")
for k, v in cfg.__dict__.items():
    print(f"  {k:>15} = {v}")

## Step 2: A Longer Corpus

We need more data than Day 14's Shakespeare snippet. Let's use a longer chunk.

In [ ]:
# Bigger corpus — mix of Shakespeare-style + repetitive structures
# (more data = better learning. Real GPT trains on trillions of tokens.)

text = """to be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles
and by opposing end them to die to sleep
no more and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to tis a consummation
devoutly to be wished to die to sleep
to sleep perchance to dream ay there's the rub
for in that sleep of death what dreams may come
when we have shuffled off this mortal coil
must give us pause there's the respect
that makes calamity of so long life

friends romans countrymen lend me your ears
i come to bury caesar not to praise him
the evil that men do lives after them
the good is oft interred with their bones
so let it be with caesar the noble brutus
hath told you caesar was ambitious
if it were so it was a grievous fault
and grievously hath caesar answer'd it

now is the winter of our discontent
made glorious summer by this sun of york
and all the clouds that lour'd upon our house
in the deep bosom of the ocean buried
now are our brows bound with victorious wreaths
our bruised arms hung up for monuments
our stern alarums changed to merry meetings
our dreadful marches to delightful measures
grim visaged war hath smoothed his wrinkled front
and now instead of mounting barbed steeds
to fright the souls of fearful adversaries
he capers nimbly in a lady's chamber
to the lascivious pleasing of a lute"""

print(f"Corpus length: {len(text)} characters")
print(f"Lines: {len(text.split(chr(10)))}")

## Step 3: Tokenizer Class

Reusable — wraps encode, decode, save, load.

In [ ]:
class CharTokenizer:
    """Character-level tokenizer with save/load."""
    
    def __init__(self, text=None, char_to_idx=None):
        if text is not None:
            chars = sorted(set(text))
            self.char_to_idx = {c: i for i, c in enumerate(chars)}
        else:
            self.char_to_idx = char_to_idx
        self.idx_to_char = {i: c for c, i in self.char_to_idx.items()}
    
    @property
    def vocab_size(self):
        return len(self.char_to_idx)
    
    def encode(self, text):
        return [self.char_to_idx[c] for c in text if c in self.char_to_idx]
    
    def decode(self, ids):
        return ''.join(self.idx_to_char[int(i)] for i in ids)
    
    def save(self, path):
        with open(path, 'w') as f:
            json.dump(self.char_to_idx, f)
    
    @classmethod
    def load(cls, path):
        with open(path, 'r') as f:
            return cls(char_to_idx=json.load(f))


tokenizer = CharTokenizer(text)
cfg.vocab_size = tokenizer.vocab_size

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"\nDemo:")
sample = "to be or not to be"
ids = tokenizer.encode(sample)
print(f"  '{sample}' → {ids[:10]}...")
print(f"  Decoded: '{tokenizer.decode(ids)}'")

## Step 4: Train / Val Split and Batch Sampler

Important: split by **position**, not by random sample. We want to validate on tokens the model has never seen.

In [ ]:
# Encode the whole text, then split

data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data = data[:n_train]
val_data   = data[n_train:]

print(f"Total tokens: {len(data):,}")
print(f"  Train: {len(train_data):,}")
print(f"  Val:   {len(val_data):,}")

def get_batch(split):
    """Sample a random batch from the chosen split."""
    d = train_data if split == 'train' else val_data
    starts = torch.randint(0, len(d) - cfg.block_size, (cfg.batch_size,))
    x = torch.stack([d[s : s + cfg.block_size] for s in starts])
    y = torch.stack([d[s+1 : s + cfg.block_size + 1] for s in starts])
    return x, y

# Try it
xb, yb = get_batch('train')
print(f"\nBatch shapes: x={xb.shape}, y={yb.shape}")
print(f"\nFirst sample (decoded):")
print(f"  Input:  '{tokenizer.decode(xb[0].tolist())}'")
print(f"  Target: '{tokenizer.decode(yb[0].tolist())}'")
print(f"\nNotice: target is input shifted by 1 character (next-token prediction).")

## Step 5: The Model — Token Embedding + Position Embedding + Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head causal self-attention (from Day 16)."""
    
    def __init__(self, embed_dim, num_heads, block_size):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_o = nn.Linear(embed_dim, embed_dim, bias=False)
        
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size)))
    
    def forward(self, x):
        B, T, C = x.shape
        H, D = self.num_heads, self.head_dim
        
        Q = self.W_q(x).view(B, T, H, D).transpose(1, 2)
        K = self.W_k(x).view(B, T, H, D).transpose(1, 2)
        V = self.W_v(x).view(B, T, H, D).transpose(1, 2)
        
        scores = Q @ K.transpose(-2, -1) / (D ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_o(out)


class AttentionLM(nn.Module):
    """Complete attention-based language model.
    
    Combines: token embedding + learned position embedding + multi-head 
    causal self-attention + output projection.
    """
    
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_emb = nn.Embedding(config.block_size, config.embed_dim)
        self.attn = MultiHeadAttention(config.embed_dim, config.num_heads, config.block_size)
        self.head = nn.Linear(config.embed_dim, config.vocab_size)
    
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        
        x = self.tok_emb(idx) + self.pos_emb(pos)   # (B, T, D)
        x = self.attn(x)                              # (B, T, D)
        logits = self.head(x)                         # (B, T, V)
        
        loss = None
        if targets is not None:
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.view(B*T, V), targets.view(B*T))
        return logits, loss

# Build the model
torch.manual_seed(42)
model = AttentionLM(cfg)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 6: Train With Proper Val-Loss Tracking

We'll periodically compute val loss to make sure we're not overfitting, and keep the best version.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    """Average loss over several random batches on each split."""
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = []
        for _ in range(cfg.eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = float(np.mean(losses))
    model.train()
    return out


# The training loop

torch.manual_seed(42)
model = AttentionLM(cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate)

train_losses_log = []
val_losses_log = []
log_steps = []

best_val_loss = float('inf')
best_state = None

for step in range(cfg.steps):
    # Eval every once in a while
    if step % cfg.eval_every == 0 or step == cfg.steps - 1:
        loss_dict = estimate_loss(model)
        train_losses_log.append(loss_dict['train'])
        val_losses_log.append(loss_dict['val'])
        log_steps.append(step)
        
        # Save best model
        if loss_dict['val'] < best_val_loss:
            best_val_loss = loss_dict['val']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        
        print(f"Step {step:4d}: train={loss_dict['train']:.4f}, val={loss_dict['val']:.4f}")
    
    # Training step
    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"\nBest val loss: {best_val_loss:.4f}")
print(f"Loading best model state...")
model.load_state_dict(best_state)

In [ ]:
# Visualize training progress + perplexity

train_perp = np.exp(train_losses_log)
val_perp = np.exp(val_losses_log)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(log_steps, train_losses_log, 'b-o', label='Train', markersize=4)
ax1.plot(log_steps, val_losses_log, 'r-o', label='Val', markersize=4)
ax1.set_xlabel('Step')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training progress (lower = better)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(log_steps, train_perp, 'b-o', label='Train', markersize=4)
ax2.plot(log_steps, val_perp, 'r-o', label='Val', markersize=4)
ax2.axhline(y=cfg.vocab_size, color='gray', linestyle='--', label=f'Random baseline ({cfg.vocab_size})')
ax2.set_xlabel('Step')
ax2.set_ylabel('Perplexity')
ax2.set_title('Perplexity (lower = better)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Random baseline perplexity: {cfg.vocab_size}")
print(f"Final train perplexity:     {train_perp[-1]:.2f}")
print(f"Final val perplexity:       {val_perp[-1]:.2f}")

## Step 7: Generation With Temperature and Top-K Sampling

A real generator gives you knobs to control creativity vs determinism.

In [ ]:
@torch.no_grad()
def generate(model, tokenizer, prompt='to ', max_new=200,
             temperature=1.0, top_k=None):
    """Generate text autoregressively.
    
    temperature < 1.0 → sharper, more deterministic
    temperature > 1.0 → flatter, more creative
    top_k = N        → only sample from N most likely tokens (more grammatical)
    top_k = None     → sample from full distribution
    """
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long)
    
    for _ in range(max_new):
        # Crop input to block_size (the model's attention range)
        idx_crop = idx[:, -cfg.block_size:]
        
        logits, _ = model(idx_crop)
        last_logits = logits[0, -1, :] / temperature   # (V,)
        
        # Apply top-K: zero out everything except the top K
        if top_k is not None:
            v, _ = last_logits.topk(top_k)
            min_keep = v[-1]                            # smallest value among top-K
            last_logits = torch.where(
                last_logits < min_keep,
                torch.tensor(float('-inf')),
                last_logits
            )
        
        probs = F.softmax(last_logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx.unsqueeze(0)], dim=1)
    
    return tokenizer.decode(idx[0].tolist())


# Try different sampling strategies
print("=" * 60)
print("ARGMAX (greedy, always pick most likely)")
print("=" * 60)
torch.manual_seed(0)
print(generate(model, tokenizer, prompt='to be', temperature=0.01, max_new=150))

print("\n" + "=" * 60)
print("LOW TEMPERATURE (0.5) — conservative")
print("=" * 60)
torch.manual_seed(0)
print(generate(model, tokenizer, prompt='to be', temperature=0.5, max_new=150))

print("\n" + "=" * 60)
print("DEFAULT (temperature=1.0)")
print("=" * 60)
torch.manual_seed(0)
print(generate(model, tokenizer, prompt='to be', temperature=1.0, max_new=150))

print("\n" + "=" * 60)
print("TOP-K=5 — only sample from 5 most likely chars")
print("=" * 60)
torch.manual_seed(0)
print(generate(model, tokenizer, prompt='to be', temperature=1.0, top_k=5, max_new=150))

print("\n" + "=" * 60)
print("HIGH TEMPERATURE (1.5) — creative / wild")
print("=" * 60)
torch.manual_seed(0)
print(generate(model, tokenizer, prompt='to be', temperature=1.5, max_new=150))

## Step 8: Save & Reload the Trained Model

Production-ready artifacts: weights + tokenizer + config. Anyone with these three files can run our generator.

In [ ]:
ARTIFACTS = '/tmp/day18_generator'
os.makedirs(ARTIFACTS, exist_ok=True)

# 1. Save the model weights
torch.save(model.state_dict(), f'{ARTIFACTS}/model.pth')

# 2. Save tokenizer
tokenizer.save(f'{ARTIFACTS}/tokenizer.json')

# 3. Save config
with open(f'{ARTIFACTS}/config.json', 'w') as f:
    json.dump(cfg.__dict__, f, indent=2)

print(f"Saved to {ARTIFACTS}/:")
for fn in os.listdir(ARTIFACTS):
    size = os.path.getsize(f'{ARTIFACTS}/{fn}')
    print(f"  {fn:>20}  ({size:,} bytes)")

In [ ]:
class TextGenerator:
    """Load from artifacts, then generate text. Production-style API."""
    
    def __init__(self, artifacts_dir):
        # Load config
        with open(f'{artifacts_dir}/config.json') as f:
            cfg_dict = json.load(f)
        self.cfg = Config(**cfg_dict)
        
        # Load tokenizer
        self.tokenizer = CharTokenizer.load(f'{artifacts_dir}/tokenizer.json')
        
        # Build and load model
        self.model = AttentionLM(self.cfg)
        self.model.load_state_dict(torch.load(f'{artifacts_dir}/model.pth'))
        self.model.eval()
    
    @torch.no_grad()
    def generate(self, prompt, max_new=200, temperature=1.0, top_k=None):
        idx = torch.tensor([self.tokenizer.encode(prompt)], dtype=torch.long)
        for _ in range(max_new):
            idx_crop = idx[:, -self.cfg.block_size:]
            logits, _ = self.model(idx_crop)
            last = logits[0, -1, :] / temperature
            if top_k is not None:
                v, _ = last.topk(top_k)
                last = torch.where(last < v[-1], torch.tensor(float('-inf')), last)
            probs = F.softmax(last, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx.unsqueeze(0)], dim=1)
        return self.tokenizer.decode(idx[0].tolist())

# Test by loading fresh
gen = TextGenerator(ARTIFACTS)
print("Loaded model from disk. Generating with top_k=5, temperature=0.9:\n")
torch.manual_seed(7)
print(gen.generate(prompt='the king', max_new=200, temperature=0.9, top_k=5))

## Step 9: Inspect Attention on a Real Sequence

Let's see what the trained model learned to attend to.

In [ ]:
# Pull out attention weights for a known phrase

import matplotlib.pyplot as plt

sample = "to be or not to be that"  # 23 chars (must be <= block_size=32)
ids = torch.tensor([tokenizer.encode(sample)], dtype=torch.long)

# Manually do the forward pass to grab the attention weights
model.eval()
with torch.no_grad():
    B, T = ids.shape
    pos = torch.arange(T)
    x = model.tok_emb(ids) + model.pos_emb(pos)
    
    # Reach inside the attention to extract weights
    H, D = model.attn.num_heads, model.attn.head_dim
    Q = model.attn.W_q(x).view(B, T, H, D).transpose(1, 2)
    K = model.attn.W_k(x).view(B, T, H, D).transpose(1, 2)
    scores = Q @ K.transpose(-2, -1) / (D ** 0.5)
    scores = scores.masked_fill(model.attn.mask[:T, :T] == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)            # (1, H, T, T)

# Plot all heads side by side
fig, axes = plt.subplots(1, cfg.num_heads, figsize=(20, 5))
for h in range(cfg.num_heads):
    ax = axes[h]
    ax.imshow(weights[0, h].numpy(), cmap='Blues')
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels([repr(c) for c in sample], fontsize=7)
    ax.set_yticklabels([repr(c) for c in sample], fontsize=7)
    ax.set_title(f'Head {h+1}')

plt.suptitle(f"Attention patterns on '{sample}'", fontsize=14)
plt.tight_layout()
plt.show()

print("Each head learned a different attention pattern.")
print("Brighter cells = stronger attention.")
print("Some heads focus on previous chars, others on specific positions.")

---

## Exercises

1. **Bigger model:** Double `embed_dim` to 128 and `num_heads` to 8. Does val loss go lower?

2. **Bigger context:** Set `block_size=64`. Does the model write better long-range text?

3. **Compare to Day 14 bigram:** On the same corpus, the Day 14 bigram model's val loss was around ~2.0. What's yours? Why is it lower?

4. **Add top-P sampling:** Implement nucleus sampling — sample from the smallest set of tokens whose total probability ≥ P.

5. **Custom corpus:** Replace the Shakespeare text with song lyrics, code, or a book you like. Train the model on it. What does it generate?

---

## What You've Built

You have a **complete, working text generator** that:

- Takes any text corpus
- Tokenizes at the character level
- Uses position-aware multi-head attention
- Trains end-to-end with proper train/val
- Can be saved to disk and loaded later
- Generates new text with configurable temperature and top-K

This is the structure of every modern LLM. The differences from here to GPT:

| You have today | GPT additionally has |
|----------------|----------------------|
| 1 attention layer | 12+ stacked transformer blocks |
| Char-level tokenizer | BPE/SentencePiece tokenizer |
| Hundreds of training tokens | Trillions of training tokens |
| 1 attention "type" | LayerNorm + Residual + MLP per block |
| Single GPU minutes | Cluster-weeks of GPUs |

The HARD ideas are done. The rest is mostly scale.

### Where we are

```
Phase 3 (sequence models)
├── Day 13: RNN                              ✓
├── Day 14: Bigram LM                        ✓
├── Day 15: Self-attention                   ✓
├── Day 16: Multi-head attention             ✓
├── Day 17: Positional encoding              ✓
└── Day 18: PROJECT — attention generator    ✓ ← YOU JUST FINISHED

Phase 4 (transformers)
├── Day 19: Transformer block (LayerNorm + MLP + residual)
├── Day 20: Mini GPT (stack of blocks)
├── Day 21: BPE tokenization
├── Day 22: Train mini GPT for real
├── Day 23: Sampling strategies
├── Day 24: Evaluation
└── Day 25: PROJECT — your own GPT on custom data
```

**Tomorrow:** Day 19 — the transformer block. We'll wrap our attention + position encoding into the canonical "block" with layer norm, residual connections, and an MLP layer. After that, "transformer" just means "stack of blocks."